In [0]:
from pyspark.sql.functions import col, avg, min, max, when, sum, count, row_number, to_date, datediff, floor, round
from pyspark.sql.window import Window

In [0]:
drivers = spark.read.option("header", True).csv("/Volumes/gr5069/raw/f1_data/drivers.csv")
results = spark.read.option("header", True).csv("/Volumes/gr5069/raw/f1_data/results.csv")
races = spark.read.option("header", True).csv("/Volumes/gr5069/raw/f1_data/races.csv")
pit_stops = spark.read.option("header", True).csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv")

In [0]:
pit_stops_clean = pit_stops \
    .withColumn("raceId", col("raceId").cast("int")) \
    .withColumn("driverId", col("driverId").cast("int")) \
    .withColumn("milliseconds", col("milliseconds").cast("int"))

results_clean = results \
    .withColumn("raceId", col("raceId").cast("int")) \
    .withColumn("driverId", col("driverId").cast("int")) \
    .withColumn("positionOrder", col("positionOrder").cast("int"))

drivers_clean = drivers \
    .withColumn("driverId", col("driverId").cast("int")) \
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))

races_clean = races \
    .withColumn("raceId", col("raceId").cast("int")) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

# Question 1: Average pit stop time for each driver in each race

To answer this question, I used the `pit_stops` dataset because it records each pit stop event by race and driver, including the duration in milliseconds. I first grouped the data by `raceId` and `driverId` to calculate the average time each driver spent at the pit stop in a given race. Then, to identify the fastest and slowest pit stop in each race, I grouped the same dataset by `raceId` only and calculated the minimum and maximum pit stop durations. Finally, I joined these results with the `drivers` and `races` datasets so that the output would include driver names and race names instead of only IDs.

In [0]:
from pyspark.sql.functions import avg, min, max, round, col

avg_pit_by_driver_race = pit_stops_clean.groupBy("raceId", "driverId").agg(
    round(avg("milliseconds") / 1000, 2).alias("avg_pit_seconds")
)

race_fastest_slowest_pit = pit_stops_clean.groupBy("raceId").agg(
    round(min("milliseconds") / 1000, 2).alias("fastest_pit_seconds"),
    round(max("milliseconds") / 1000, 2).alias("slowest_pit_seconds")
)

q1_result = avg_pit_by_driver_race.join(
    race_fastest_slowest_pit,
    on="raceId",
    how="left"
).join(
    drivers_clean.select("driverId", "forename", "surname", "code"),
    on="driverId",
    how="left"
).join(
    races_clean.select("raceId", "name", "date"),
    on="raceId",
    how="left"
).select(
    "raceId",
    "name",
    "date",
    "driverId",
    "forename",
    "surname",
    "code",
    "avg_pit_seconds",
    "fastest_pit_seconds",
    "slowest_pit_seconds"
).orderBy("raceId", "avg_pit_seconds")

display(q1_result)

I used `groupBy("raceId", "driverId")` together with `avg("milliseconds")` to calculate the average pit stop duration for each driver in each race. Since the original values were recorded in milliseconds, I divided them by 1000 and used `round(..., 2)` to express the results in seconds with two decimal places.

Next, I used a second aggregation with `groupBy("raceId")` to calculate the fastest and slowest pit stop in each race using `min("milliseconds")` and `max("milliseconds")`. This produces one row per race that summarizes the range of pit stop durations during that event.

After that, I joined the driver-level averages to the race-level minimum and maximum values using `join(..., on="raceId", how="left")`. I then joined the result to the `drivers_clean` and `races_clean` tables so that the final output includes descriptive fields such as driver names, race names, and dates. Finally, I used `select()` to keep the relevant columns and `orderBy()` to sort the table by race and average pit stop time.

In [0]:
display(race_fastest_slowest_pit.orderBy("raceId"))

# Question 2: Rank order the average pit stop time by finishing position in each race

To answer this question, I used the average pit stop time calculated in Question 1 and joined it with the finishing results from the `results` dataset. My goal was to compare pit stop performance with race outcome, so I matched each driver's average pit stop time in a given race to that driver's finishing position in the same race.

I used `positionOrder` from the `results` table as the ranking variable because it provides a consistent official ordering of drivers within each race. For drivers who did not finish the race, I kept their recorded `positionOrder` if it was available in the dataset. I chose this approach because the results table already reflects the race classification and avoids imposing my own ranking rule.

In [0]:
q2_result = avg_pit_by_driver_race.join(
    results_clean.select("raceId", "driverId", "positionOrder", "positionText"),
    on=["raceId", "driverId"],
    how="inner"
).join(
    drivers_clean.select("driverId", "forename", "surname", "code"),
    on="driverId",
    how="left"
).join(
    races_clean.select("raceId", "name", "date"),
    on="raceId",
    how="left"
).select(
    "raceId",
    "name",
    "date",
    "positionOrder",
    "positionText",
    "driverId",
    "forename",
    "surname",
    "code",
    "avg_pit_seconds"
).orderBy("raceId", "positionOrder")

display(q2_result)

In [0]:
display(q2_result.limit(10))

I began with the table of average pit stop times created in Question 1, which contains one row per driver per race. I then joined this table to the `results_clean` dataset using both `raceId` and `driverId` so that each average pit stop value would be matched to the correct finishing result.

I selected `positionOrder` and `positionText` from the results table to preserve both the numeric rank and the original result label. The command `orderBy("raceId", "positionOrder")` sorts drivers within each race according to their official classification, which makes it possible to compare pit stop averages against finishing position.

I used an inner join because I only wanted observations that appear in both the pit stop data and the race results. After that, I joined the output to the `drivers_clean` and `races_clean` tables so that the final table includes readable driver and race information rather than IDs alone.

# Question 3: Insert the missing three-letter driver codes

To answer this question, I first identified all drivers whose `code` value was missing or stored as the placeholder `\N` in the `drivers` dataset. After checking the existing non-missing values, I observed that driver codes generally follow a three-letter uppercase format that is closely tied to the driver's surname.

Because there were many missing entries, I used a consistent surname-based rule to generate replacement codes rather than filling them manually one by one. My rule was to remove punctuation and non-letter characters from the surname, convert the result to uppercase, and then take the first three letters. I kept all existing non-missing codes unchanged. I chose this method because it is systematic, reproducible, and consistent with the structure of the existing code variable.

In [0]:
from pyspark.sql.functions import regexp_replace, upper, substring

drivers_filled = drivers_clean.withColumn(
    "code",
    when(
        col("code").isNull() | (col("code") == "") | (col("code") == "\\N"),
        upper(
            substring(
                regexp_replace(col("surname"), "[^A-Za-z]", ""),
                1,
                3
            )
        )
    ).otherwise(col("code"))
)

display(
    drivers_filled.select("driverId", "forename", "surname", "code").orderBy("driverId")
)

In [0]:
remaining_missing_codes = drivers_filled.filter(
    col("code").isNull() | (col("code") == "") | (col("code") == "\\N")
).select("driverId", "forename", "surname", "code")

display(remaining_missing_codes)

In [0]:
display(
    drivers_filled.filter(col("driverId").between(50, 80))
    .select("driverId", "forename", "surname", "code")
    .orderBy("driverId")
)

In [0]:
original_missing = drivers_clean.filter(col("code") == "\\N").select(
    "driverId", "forename", "surname"
)

filled_preview = original_missing.join(
    drivers_filled.select("driverId", "code"),
    on="driverId",
    how="left"
).orderBy("driverId")

display(filled_preview)


I first filtered the `drivers_clean` table to identify records where `code` was null, blank, or equal to the placeholder `\N`. This showed that the missing values in this dataset were primarily stored as placeholders rather than as ordinary null values.

To fill these missing entries, I used `withColumn("code", ...)` together with a `when()` condition. For records with missing codes, I applied `regexp_replace()` to the `surname` column to remove punctuation and non-letter characters, then used `substring(..., 1, 3)` to extract the first three letters and `upper()` to convert them to uppercase. This created a consistent three-letter surname-based code for each missing case. The command `otherwise(col("code"))` preserved all existing non-missing values.

Finally, I validated the result by filtering the updated table again for null, blank, and `\N` values in the `code` column. Since no rows were returned, the replacement procedure successfully removed all missing placeholders.

# Question 4: Identify the youngest and oldest driver in each race

To answer this question, I first created a driver-race level table by joining the `results`, `drivers`, and `races` datasets. I defined a driver's age as the number of full birthdays the driver had already had on the date of the race. To calculate this, I used the difference between the race date and the driver's date of birth and converted that value into years.

After creating the `Age` column, I identified the youngest and oldest driver in each race by ranking drivers within each race according to age. I used one ranking in ascending order to find the youngest driver and another ranking in descending order to find the oldest driver.

In [0]:
driver_race_age = results_clean.select("raceId", "driverId").join(
    drivers_filled.select("driverId", "forename", "surname", "dob", "code"),
    on="driverId",
    how="left"
).join(
    races_clean.select("raceId", "name", "date"),
    on="raceId",
    how="left"
).withColumn(
    "Age",
    floor(datediff(col("date"), col("dob")) / 365.25)
)

youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

youngest_driver = driver_race_age.withColumn(
    "youngest_rank",
    row_number().over(youngest_window)
).filter(
    col("youngest_rank") == 1
).select(
    "raceId", "name", "date", "driverId", "forename", "surname", "code", "Age"
).withColumnRenamed("driverId", "youngest_driverId") \
 .withColumnRenamed("forename", "youngest_forename") \
 .withColumnRenamed("surname", "youngest_surname") \
 .withColumnRenamed("code", "youngest_code") \
 .withColumnRenamed("Age", "youngest_age")

oldest_driver = driver_race_age.withColumn(
    "oldest_rank",
    row_number().over(oldest_window)
).filter(
    col("oldest_rank") == 1
).select(
    "raceId", "driverId", "forename", "surname", "code", "Age"
).withColumnRenamed("driverId", "oldest_driverId") \
 .withColumnRenamed("forename", "oldest_forename") \
 .withColumnRenamed("surname", "oldest_surname") \
 .withColumnRenamed("code", "oldest_code") \
 .withColumnRenamed("Age", "oldest_age")

q4_result = youngest_driver.join(
    oldest_driver,
    on="raceId",
    how="inner"
).orderBy("raceId")

display(q4_result)

I began by joining `results_clean`, `drivers_filled`, and `races_clean` so that each observation would represent one driver in one race, along with the driver's birth date and the race date. I then created the `Age` column using `datediff(col("date"), col("dob"))`, which calculates the number of days between the race date and the driver's date of birth. I divided this by `365.25` and applied `floor()` to represent the number of full birthdays the driver had already had by the race date.

To identify the youngest and oldest drivers in each race, I used window functions. The window `Window.partitionBy("raceId").orderBy(col("Age").asc())` ranks drivers from youngest to oldest within each race, while the descending version ranks them from oldest to youngest. I used `row_number()` to assign ranks and filtered for rank 1 in each case.

Finally, I joined the youngest-driver table and the oldest-driver table by `raceId` so that the final output shows both results side by side for each race. Because I used `row_number()`, if multiple drivers had the exact same age in a race, only one of them would be retained as rank 1. In this notebook, I accepted that tie-breaking rule in order to produce one youngest and one oldest driver per race.

# Question 5: Count prior wins, second places, and third places for each driver at each race

To answer this question, I used the `results` dataset because it records each driver's finishing position in every race. I created three indicator variables to show whether a driver finished first, second, or third in a race. Then I calculated, for each driver and each race, how many wins, second places, and third places the driver had already accumulated before that race.

I used a window function partitioned by `driverId` and ordered by race date so that the counts would be accumulated in chronological order within each driver's career. I defined the window to stop at the row immediately before the current race because the question asks for the number of podium finishes a driver had at a given race, which I interpret as the total achieved before that race's result is counted.

In [0]:
from pyspark.sql.functions import when, sum, col, coalesce, lit
from pyspark.sql.window import Window

podium_base = results_clean.select("raceId", "driverId", "positionOrder").join(
    races_clean.select("raceId", "date", "name"),
    on="raceId",
    how="left"
).withColumn(
    "win_flag",
    when(col("positionOrder") == 1, 1).otherwise(0)
).withColumn(
    "second_flag",
    when(col("positionOrder") == 2, 1).otherwise(0)
).withColumn(
    "third_flag",
    when(col("positionOrder") == 3, 1).otherwise(0)
)

podium_window = Window.partitionBy("driverId").orderBy("date").rowsBetween(Window.unboundedPreceding, -1)

q5_result = podium_base.withColumn(
    "prior_wins",
    coalesce(sum("win_flag").over(podium_window), lit(0))
).withColumn(
    "prior_second_places",
    coalesce(sum("second_flag").over(podium_window), lit(0))
).withColumn(
    "prior_third_places",
    coalesce(sum("third_flag").over(podium_window), lit(0))
).join(
    drivers_filled.select("driverId", "forename", "surname", "code"),
    on="driverId",
    how="left"
).select(
    "raceId",
    "name",
    "date",
    "driverId",
    "forename",
    "surname",
    "code",
    "positionOrder",
    "prior_wins",
    "prior_second_places",
    "prior_third_places"
).orderBy("raceId", "positionOrder")

display(q5_result)

In [0]:
display(
    q5_result.filter(col("driverId") == 1).orderBy("date").limit(10)
)

I began by joining `results_clean` with `races_clean` so that each result would include the race date. This was necessary because prior podium counts must be calculated in chronological order. I then created three binary indicator columns with `when()`: `win_flag` equals 1 when `positionOrder` is 1, `second_flag` equals 1 when `positionOrder` is 2, and `third_flag` equals 1 when `positionOrder` is 3. All other rows were assigned 0.

Next, I defined a window with `Window.partitionBy("driverId").orderBy("date").rowsBetween(Window.unboundedPreceding, -1)`. This groups observations within each driver's career, sorts them by race date, and includes all prior races while excluding the current one. That choice matches the interpretation that the question asks for the number of podium finishes a driver had before the current race result is counted.

I then used `sum(...).over(podium_window)` to calculate cumulative prior wins, prior second places, and prior third places. Because the first race for each driver has no earlier rows, I used `coalesce(..., lit(0))` to replace null values with 0. Finally, I joined the output to `drivers_filled` to add driver names and codes, selected the relevant columns, and sorted the final table by race and finishing order.

# Question 6: Which race had the highest average pit stop time across all drivers?

For my own question, I asked which race had the highest average pit stop time across all drivers. I chose this question because pit stop duration is an important part of race strategy, and the `pit_stops` dataset allows me to compare overall pit stop performance across races.

To answer it, I grouped pit stop observations by race, calculated the average pit stop duration in seconds for each race, and then joined the result to the `races` dataset so that the output would include race names and dates. Finally, I sorted the races from highest to lowest average pit stop time.

In [0]:
q6_result = pit_stops_clean.groupBy("raceId").agg(
    round(avg("milliseconds") / 1000, 2).alias("race_avg_pit_seconds")
).join(
    races_clean.select("raceId", "name", "date"),
    on="raceId",
    how="left"
).select(
    "raceId",
    "name",
    "date",
    "race_avg_pit_seconds"
).orderBy(col("race_avg_pit_seconds").desc())

display(q6_result)

In [0]:
display(q6_result.limit(10))

I used `groupBy("raceId")` on the `pit_stops_clean` dataset to aggregate all pit stop records within each race. Then I applied `avg("milliseconds")` to compute the mean pit stop duration and converted the result from milliseconds to seconds by dividing by 1000. I used `round(..., 2)` so the values would be easier to read.

Next, I joined the aggregated result to `races_clean` using `raceId` so that the final table would include race names and dates instead of only numeric IDs. Finally, I used `orderBy(col("race_avg_pit_seconds").desc())` to rank the races from the highest average pit stop time to the lowest.